# HW4 Q1: Post-training quantization (PTQ) of ResNet-20
Quantize the weights of every `Conv2d` layer of the ResNet-20 trained in HW3 to **8 bits** and **4 bits**, then evaluate on the CIFAR-10 test set.
Only the weights change: no activation quantization and no retraining.

**Flow (same as HW3 Part I, with PTQ in the middle):** reload the checkpoint → quantize the conv weights → run the same `validate()`.

Needs the HW3 checkpoint `result/resnet20/model_best.pth.tar` (this notebook sits next to the HW3 notebooks in `software/hw/`).

## Setup (same as HW3)

In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

    

import torchvision
import torchvision.transforms as transforms

import sys
sys.path.insert(0, os.path.abspath('..'))   # this notebook is in software/hw/, the models/ folder is in software/
from models import *   # bring everything in the folder models

global best_prec
use_gpu = torch.cuda.is_available()
device = 'cuda' if use_gpu else 'cpu'   # Q1 only runs inference, so it also works on a CPU-only server
print('device:', device)
print('=> Building model...')
    
    
batch_size = 128

model_name = "resnet20"
model = resnet20_cifar()

        

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.to(device), target.to(device)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.to(device), target.to(device)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.view(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

device: cuda
=> Building model...
Files already downloaded and verified
Files already downloaded and verified


## FP32 baseline: the HW3 checkpoint, unquantized

In [2]:
criterion = nn.CrossEntropyLoss().to(device)
ckpt_path = 'result/'+str(model_name)+'/model_best.pth.tar'

def load_trained_model():
    # Build a fresh model and load the HW3 checkpoint. Called once per experiment, so every
    # quantized model starts from the same FP32 weights (never quantize already-quantized weights).
    model = resnet20_cifar()
    checkpoint = torch.load(ckpt_path, map_location=device)   # map_location: the checkpoint was saved from a GPU
    model.load_state_dict(checkpoint['state_dict'])
    model.to(device)
    model.eval()
    return model

model = load_trained_model()
n_conv = sum(isinstance(m, nn.Conv2d) for m in model.modules())
print('Conv2d layers to quantize:', n_conv)
fp32_acc = validate(testloader, model, criterion)

Conv2d layers to quantize: 21
Test: [0/79]	Time 0.372 (0.372)	Loss 0.1580 (0.1580)	Prec 94.531% (94.531%)
 * Prec 92.190% 


## Weight quantizer (same steps as `weight_quantize_fn()` in [Code8])
1. divide by the clipping value $\alpha$
2. clip to $[-1, 1]$
3. quantize the magnitude uniformly to $2^{b-1}-1$ levels and put the sign back (1 bit is the sign, like `self.w_bit = w_bit-1` in [Code8])
4. rescale by $\alpha$

So the quantized weights are integers times the resolution $\Delta = \alpha / (2^{b-1}-1)$:
8-bit → integers in $[-127, 127]$, 4-bit → integers in $[-7, 7]$.

**Choosing $\alpha$** (one $\alpha$ per layer), two options compared below:
- **max |w|**: $\alpha$ = the largest weight magnitude in the layer. Nothing gets clipped, but a few outlier weights stretch the range, so the step $\Delta$ is large for the many small weights.
- **MSE-optimal**: search $\alpha = r \cdot \max|w|$ for $r \in [0.05, 1]$ and keep the one with the smallest mean squared error between the quantized and original weights. This trades some clipping error on the few large weights for a finer step on all the others.

In [3]:
def quantize_weight(w, alpha, w_bit):
    # Same steps as weight_quantization() / weight_quantize_fn() in [Code8].
    # (Written as a plain function: [Code8]'s uq class is an old-style autograd.Function,
    #  which newer PyTorch versions refuse to run, and PTQ needs no gradients anyway.)
    b = w_bit - 1                                # 1 bit for the sign, as in [Code8]: self.w_bit = w_bit-1
    n_levels = 2 ** b - 1                        # 127 for 8-bit, 7 for 4-bit
    w_d = w / alpha                              # 1. divide by the clipping value alpha
    w_c = w_d.clamp(min=-1, max=1)               # 2. clip to [-1, 1]
    sign = w_c.sign()
    w_q = (w_c.abs() * n_levels).round().div(n_levels).mul(sign)   # 3. uniform quantization of |w|, sign put back
    return w_q * alpha                           # 4. rescale to the original range


def alpha_max(w, w_bit):
    return w.abs().max()


def alpha_mse(w, w_bit, n_grid=100):
    w_max = w.abs().max()
    best_alpha, best_err = w_max, float('inf')
    for r in torch.linspace(0.05, 1.0, n_grid):
        alpha = r * w_max
        err = (quantize_weight(w, alpha, w_bit) - w).pow(2).mean().item()
        if err < best_err:
            best_alpha, best_err = alpha, err
    return best_alpha


def apply_ptq(model, w_bit, choose_alpha):
    # Quantize the weights of every Conv2d in place. no_grad: the weights are parameters
    # with requires_grad=True, so PyTorch would reject an in-place update that it tracks.
    alphas = []
    with torch.no_grad():
        for layer in model.modules():
            if isinstance(layer, torch.nn.Conv2d):
                alpha = choose_alpha(layer.weight, w_bit)
                layer.weight.copy_(quantize_weight(layer.weight, alpha, w_bit))
                alphas.append(alpha)
    return alphas


def check_quantized(model, alphas, w_bit):
    # Sanity check from the handout: weight / resolution must be (close to) an integer.
    n_levels = 2 ** (w_bit - 1) - 1
    convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    worst, n_unique, lo, hi = 0.0, 0, 0, 0
    with torch.no_grad():
        for layer, alpha in zip(convs, alphas):
            w_int = layer.weight / (alpha / n_levels)
            worst = max(worst, (w_int - w_int.round()).abs().max().item())
            n_unique = max(n_unique, w_int.round().unique().numel())
            lo, hi = min(lo, int(w_int.round().min())), max(hi, int(w_int.round().max()))
    print(f'  weight/resolution: max distance from an integer = {worst:.2e}, '
          f'integer range [{lo}, {hi}], at most {n_unique} distinct values per layer '
          f'(allowed: {2 * n_levels + 1})')

## Quantize to 8 and 4 bits and evaluate

In [4]:
results, all_alphas = {}, {}
for w_bit in [8, 4]:
    for alpha_name, choose_alpha in [('max |w|', alpha_max), ('MSE-optimal', alpha_mse)]:
        print(f'===== {w_bit}-bit weights, alpha = {alpha_name}')
        model = load_trained_model()                     # 1. reload the checkpoint
        alphas = apply_ptq(model, w_bit, choose_alpha)   # 2. PTQ on the conv weights
        check_quantized(model, alphas, w_bit)
        results[(w_bit, alpha_name)] = validate(testloader, model, criterion)   # 3. same test as HW3
        all_alphas[(w_bit, alpha_name)] = alphas

===== 8-bit weights, alpha = max |w|
  weight/resolution: max distance from an integer = 1.53e-05, integer range [-127, 127], at most 219 distinct values per layer (allowed: 255)
Test: [0/79]	Time 0.094 (0.094)	Loss 0.1606 (0.1606)	Prec 94.531% (94.531%)
 * Prec 92.140% 
===== 8-bit weights, alpha = MSE-optimal
  weight/resolution: max distance from an integer = 1.53e-05, integer range [-127, 127], at most 228 distinct values per layer (allowed: 255)
Test: [0/79]	Time 0.099 (0.099)	Loss 0.1669 (0.1669)	Prec 94.531% (94.531%)
 * Prec 92.080% 
===== 4-bit weights, alpha = max |w|
  weight/resolution: max distance from an integer = 4.77e-07, integer range [-7, 7], at most 15 distinct values per layer (allowed: 15)
Test: [0/79]	Time 0.109 (0.109)	Loss 0.3572 (0.3572)	Prec 87.500% (87.500%)
 * Prec 88.110% 
===== 4-bit weights, alpha = MSE-optimal
  weight/resolution: max distance from an integer = 4.77e-07, integer range [-7, 7], at most 15 distinct values per layer (allowed: 15)
Test: [0/

## Example: the first conv filter as integers (4-bit, MSE-optimal α)

In [5]:
model = load_trained_model()
alphas = apply_ptq(model, 4, alpha_mse)
first_conv = model.conv1
delta = alphas[0] / (2 ** (4 - 1) - 1)
print('alpha =', round(alphas[0].item(), 4), '  resolution =', round(delta.item(), 4))
print('first filter / resolution:')
print((first_conv.weight[0] / delta).round().int().cpu())

alpha = 0.6778   resolution = 0.0968
first filter / resolution:
tensor([[[ 1,  3,  1],
         [-1,  3,  0],
         [-2, -2, -1]],

        [[ 1,  4,  2],
         [-1,  2,  0],
         [-2, -3, -2]],

        [[ 0,  2,  1],
         [-1,  2,  0],
         [-1, -1, -2]]], dtype=torch.int32)


## Summary

In [6]:
print(f'{"weights":<10}{"alpha":<14}{"test acc":>10}{"vs FP32":>10}')
print(f'{"FP32":<10}{"-":<14}{fp32_acc:>9.2f}%{"":>10}')
for (w_bit, alpha_name), acc in results.items():
    print(f'{str(w_bit)+"-bit":<10}{alpha_name:<14}{acc:>9.2f}%{acc - fp32_acc:>+9.2f}%')

# how much the MSE search clips: alpha as a fraction of each layer's max |w|
model = load_trained_model()
w_max = [m.weight.abs().max() for m in model.modules() if isinstance(m, nn.Conv2d)]
for w_bit in [8, 4]:
    r = [(a / m).item() for a, m in zip(all_alphas[(w_bit, 'MSE-optimal')], w_max)]
    print(f'\n{w_bit}-bit MSE-optimal alpha / max|w| per layer: '
          f'min {min(r):.2f}, mean {sum(r)/len(r):.2f}, max {max(r):.2f}')

weights   alpha           test acc   vs FP32
FP32      -                 92.19%          
8-bit     max |w|           92.14%    -0.05%
8-bit     MSE-optimal       92.08%    -0.11%
4-bit     max |w|           88.11%    -4.08%
4-bit     MSE-optimal       86.37%    -5.82%

8-bit MSE-optimal alpha / max|w| per layer: min 0.87, mean 0.96, max 1.00

4-bit MSE-optimal alpha / max|w| per layer: min 0.40, mean 0.56, max 0.74


## Observations

the 4 bit model makes us lose some accuracy by about 4% (without retraining), scoring 88.11% vs our 92.19% accuracy for FP32. This is likely due to many small weights rounding to 0 due to low precision. 

However, the 8 bit model essentially loses no accuracy (without retraining), scoring 92.14% vs our 92.19% accuracy for FP32. This is because our 8 bits allow 127 levels per sign, compared to the 4 bit models minimal 7 levels per sign. Our step size here is tiny compared to the spread of weights, which means any rounding behaves as small random noise, and our network tolerates it well, we can therefore benefit from a 4x smaller model than FP32 with no downside by using 8 bit. 

Also, the smallest weight error is not equal to the best accuracy, MSE-optimal α scored 86.37% vs 88.11% for max |w| at 4 bits, despite lower weight error.

The MSE search minimizes how far the quantized weights are from the originals, but that doesn't perfectly align with maximizing accuracy. At 4 bits it clipped every weight above about 56% of the layer's max to ±α. This increases our clipping error, which impacts the largest most impactful weights. These weights are the most important, and this overall harms our accuracy. Rounding error on the otherhand is random, and ends up averaging out to have minimal impact to overall accuracy.

So in this network, keeping the outliers exact (max |w|) is worth more than a finer step for the small weights.

A better way to choose α would be to judge it by accuracy rather than weight error, e.g. searching the clipping ratio on held-out training data, or giving each output channel its own α, so a filter with small weights isn't forced onto a step size set by the layer's largest weight (where it could round entirely to 0).
